## Система рекомендаций игр Steam (Content-Based)

In [44]:
import json
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

### Загрузка данных

In [45]:
with open('../data/steamUsersDataset.json', encoding='utf-8') as file:
    steamUsersDataset = json.load(file)

filteredSteamUsersDataset = []
for user in steamUsersDataset:
    if not user.get('error'):
        filteredSteamUsersDataset.append(user)

print(f"Публичных пользователей: {len(filteredSteamUsersDataset)}")

with open('../data/steamGamesDataset.json', encoding='utf-8') as file:
    steamGamesDataset = json.load(file)

print(f"Игр для рекомендаций: {len(steamGamesDataset)}")

Публичных пользователей: 9978
Игр для рекомендаций: 26072


### Подготовка тегов (Game → Normalized Tags)

In [46]:
gameToNormalizedTags = {}

for game in steamGamesDataset:
    appId = game.get('appid')
    rawTags = game.get('tags', {})
    if not rawTags:
        gameToNormalizedTags[appId] = {}
        continue
    totalWeight = sum(rawTags.values())
    if totalWeight == 0:
        gameToNormalizedTags[appId] = {}
        continue
    normalized = {tag: weight / totalWeight for tag, weight in rawTags.items()}
    gameToNormalizedTags[appId] = normalized

### Построение user-tag матрицы

In [47]:
tagsNum = 100

allTagCounter = Counter()
for tags in gameToNormalizedTags.values():
    for tag in tags:
        allTagCounter[tag] += 1

topTags = [tag for tag, _ in allTagCounter.most_common(tagsNum)]
print(f"Используем {len(topTags)} топ-тегов")

userTagFeaturesList = []

for user in filteredSteamUsersDataset:
    steamId = user.get('steamId')
    tagVector = {tag: 0.0 for tag in topTags}
    totalPlaytimeHours = 0.0
    
    for game in user.get('ownedGames', []):
        appId = game.get('appid')
        playtimeMinutes = game.get('playtimeForever', 0)
        playtimeHours = playtimeMinutes / 60.0
        if playtimeHours <= 0:
            continue
        totalPlaytimeHours += playtimeHours
        normalizedTags = gameToNormalizedTags.get(appId, {})
        for tag, normWeight in normalizedTags.items():
            if tag in topTags:
                tagVector[tag] += playtimeHours * normWeight
    
    if totalPlaytimeHours > 0:
        tagVector = {tag: value / totalPlaytimeHours for tag, value in tagVector.items()}
    
    userTagFeaturesList.append({'steamId': steamId, **tagVector})

userTagFeaturesDataFrame = pd.DataFrame(userTagFeaturesList)
print("User-tag матрица построена. Размер:", userTagFeaturesDataFrame.shape)

Используем 100 топ-тегов
User-tag матрица построена. Размер: (9978, 101)


### Построение game-tag матрицы

In [48]:
gameTagList = []

for game in steamGamesDataset:
    appId = game.get('appid')
    name = game.get('name')
    normalizedTags = gameToNormalizedTags.get(appId, {})
    if not normalizedTags:
        continue
    row = {'appid': appId, 'name': name}
    for tag in topTags:
        row[tag] = normalizedTags.get(tag, 0.0)
    gameTagList.append(row)

gameTagDataFrame = pd.DataFrame(gameTagList)
print("Game-tag матрица построена. Размер:", gameTagDataFrame.shape)

Game-tag матрица построена. Размер: (22590, 102)


### Функция рекомендаций

In [49]:
def recommendGamesForUser(steamId: str, topN: int = 10):
    """
    Рекомендует topN игр пользователю по cosine similarity тегов.
    """

    if steamId not in userTagFeaturesDataFrame['steamId'].values:
        print(f"Пользователь {steamId} не найден.")
        return pd.DataFrame()
    
    userRow = userTagFeaturesDataFrame[userTagFeaturesDataFrame['steamId'] == steamId].iloc[0]
    userVector = userRow[topTags].values.reshape(1, -1)
    
    gameVectors = gameTagDataFrame[topTags].values
    
    similarities = cosine_similarity(userVector, gameVectors)[0]
    
    recommendations = gameTagDataFrame.copy()
    recommendations['similarityScore'] = similarities
    
    userOwnedAppIds = set()
    for userData in filteredSteamUsersDataset:
        if userData.get('steamId') == steamId:
            userOwnedAppIds = {g['appid'] for g in userData.get('ownedGames', [])}
            break
    
    recommendations = recommendations[~recommendations['appid'].isin(userOwnedAppIds)]
    
    topRecommendations = recommendations.nlargest(topN, 'similarityScore')
    return topRecommendations[['appid', 'name', 'similarityScore']]

### Пример рекомендаций

In [50]:
exampleSteamId = filteredSteamUsersDataset[1]['steamId']
print(f"Пример рекомендаций для пользователя {exampleSteamId}")
recs = recommendGamesForUser(exampleSteamId, topN=10)
print(recs)

Пример рекомендаций для пользователя 76561198394015316
       appid                                           name  similarityScore
4529  209160                           Call of Duty: Ghosts         0.910043
4532  209170                           Call of Duty: Ghosts         0.908606
9        300                          Day of Defeat: Source         0.903706
216    24960                     Battlefield: Bad Company 2         0.896044
472   209650  Call of Duty: Advanced Warfare - Gold Edition         0.895579
474   209660  Call of Duty: Advanced Warfare - Gold Edition         0.887567
123    10190          Call of Duty: Modern Warfare 2 (2009)         0.886821
122    10180          Call of Duty: Modern Warfare 2 (2009)         0.885102
4461   47830                                 Medal of Honor         0.883404
2185  581320                          Insurgency: Sandstorm         0.881857


### Оценка качества (Hold-out validation)

In [ ]:
# Метрики качества: Precision@K, Recall@K
def evaluateRecommendations(k: int = 10, minGames: int = 20):
    """
    - Для пользователей с >= minGames играми
    - Скрываем 20% игр как test set
    - Строим user-vector только на train-играх
    - Считаем, сколько test-игр попало в топ-K рекомендаций
    """
    precisionList = []
    recallList = []
    
    for user in filteredSteamUsersDataset:
        steamId = user.get('steamId')
        ownedGames = user.get('ownedGames', [])
        if len(ownedGames) < minGames:
            continue
        
        ownedGamesSorted = sorted(ownedGames, key=lambda g: g.get('playtimeForever', 0), reverse=True)
        trainSize = int(len(ownedGamesSorted) * 0.8)
        trainGames = ownedGamesSorted[:trainSize]
        testGames = ownedGamesSorted[trainSize:]
        
        testAppIds = {g['appid'] for g in testGames}
        
        tagVector = {tag: 0.0 for tag in topTags}
        totalTrainHours = 0.0
        
        for game in trainGames:
            appId = game['appid']
            playtimeHours = game.get('playtimeForever', 0) / 60.0
            if playtimeHours <= 0:
                continue
            totalTrainHours += playtimeHours
            normTags = gameToNormalizedTags.get(appId, {})
            for tag, weight in normTags.items():
                if tag in topTags:
                    tagVector[tag] += playtimeHours * weight
        
        if totalTrainHours == 0:
            continue
        userVector = np.array([tagVector[tag] / totalTrainHours for tag in topTags]).reshape(1, -1)
        
        gameVectors = gameTagDataFrame[topTags].values
        similarities = cosine_similarity(userVector, gameVectors)[0]
        
        trainAppIds = {g['appid'] for g in trainGames}
        recDf = gameTagDataFrame.copy()
        recDf['score'] = similarities
        recDf = recDf[~recDf['appid'].isin(trainAppIds)]
        
        topK = recDf.nlargest(k, 'score')['appid'].values
        
        hits = len(set(topK) & testAppIds)
        precision = hits / k
        recall = hits / len(testAppIds) if len(testAppIds) > 0 else 0
        
        precisionList.append(precision)
        recallList.append(recall)
    
    meanPrecision = np.mean(precisionList)
    meanRecall = np.mean(recallList)
    
    print(f"Оценка на {len(precisionList)} пользователях (min {minGames} игр)")
    print(f"Precision@{k}: {meanPrecision:.4f}")
    print(f"Recall@{k}:    {meanRecall:.4f}")
    
    return meanPrecision, meanRecall

In [32]:
evaluateRecommendations(k=10, minGames=20)

Оценка на 6359 пользователях (min 20 игр)
Precision@10: 0.0093
Recall@10:    0.0064


(np.float64(0.009293914137442995), np.float64(0.0064285376857637615))

### Сохранение модели (user и game векторы)

In [33]:
userTagFeaturesDataFrame.to_csv('../data/userTagMatrix.csv', index=False)
gameTagDataFrame.to_csv('../data/gameTagMatrix.csv', index=False)

print("Матрицы сохранены для дальнейшего использования.")

Матрицы сохранены для дальнейшего использования.
